# Which season is it? A CNN reads a sunrise

On the slides, one photo of a sunrise answers two questions: the **time of the day**, a regression, and the **season**, a classification. This notebook answers the second one. A small **convolutional neural network**, a CNN as on weekend 3, looks at a sunrise and says in which season it was taken.

The pictures are synthetic: they are drawn by the generator of the Sunrise Machine, the exercise of 09:00, written by Frederick Puls. A seeded random generator picks the moments, so every run sees the same sunrises.

**This is the worked example for the exercise "What time is it?"**, in which you build the same network for the time of the day:

https://colab.research.google.com/github/eth-bmai-hs26/w4-cx-public/blob/main/sunrise-cnn/sunrise_time_regression.ipynb

Only two cells show their code, the network in part 3 and its training in part 4: they hold the lines the exercise asks for. The other cells make the pictures, draw them and score the network, so Colab shows only their titles. Run them as they are; the Show code button opens one if you are curious.

In [ ]:
#@title Imports { display-mode: "form" }
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

## 1. The sunrises

A picture is $32 \times 32$ pixels in colour, so $32 \times 32 \times 3 = 3072$ numbers between $0$ and $1$: how red, how green and how blue each pixel is. The generator draws it from two numbers between $0$ and $1$:

* `sun_x`, where the sun stands, from left to right: the **time of the year**, from winter at $0$ to summer at $1$,
* `sun_y`, how high the sun stands: the **time of the day**, from dawn at $0$ to noon at $1$.

The colours of the sky follow the sun: indigo and gold at dawn, blue at noon, a little warmer in summer. The hills on the horizon change with the time of the year too.

In [ ]:
#@title The Sunrise Machine's generator, by Frederick Puls { display-mode: "form" }
# Written by Frederick Puls for the Sunrise Machine, the exercise of 09:00, in its file
# generate_dataset.py, and copied here with three changes that leave every picture the
# same: its loops run over lists instead of over a range, its comments are plain ASCII
# text, and its type hints are gone.
#
# render_image(sun_x, sun_y) draws one sunrise, 32 x 32 pixels in colour, from
#   sun_x, where the sun stands from left to right: the time of the year, winter 0 to summer 1,
#   sun_y, how high the sun stands: the time of the day, dawn 0 to noon 1.
import math


# ---------------------------------------------------------------------------
# Reference sky palettes: each is a list of (row_fraction, RGB) color stops.
# We define four palettes keyed by sun_y, and interpolate between them.
# ---------------------------------------------------------------------------

# Dawn  (sun_y about 0)
_PALETTE_DAWN = np.array([
    [0.08, 0.04, 0.18],  # top: dark indigo
    [0.20, 0.08, 0.28],  # deep violet
    [0.70, 0.25, 0.35],  # rose
    [0.90, 0.60, 0.20],  # gold at horizon
])

# Early morning (sun_y about 0.3)
_PALETTE_EARLY = np.array([
    [0.05, 0.05, 0.20],  # navy
    [0.15, 0.25, 0.55],  # blue
    [0.85, 0.60, 0.45],  # peach
    [0.90, 0.50, 0.15],  # orange at horizon
])

# Mid morning (sun_y about 0.7)
_PALETTE_MID = np.array([
    [0.15, 0.30, 0.65],  # blue
    [0.40, 0.60, 0.85],  # light blue
    [0.80, 0.80, 0.60],  # pale yellow
    [0.90, 0.85, 0.55],  # yellow at horizon
])

# Noon (sun_y about 1.0)
_PALETTE_NOON = np.array([
    [0.10, 0.20, 0.55],  # deep blue
    [0.30, 0.55, 0.85],  # sky blue
    [0.70, 0.80, 0.95],  # light sky
    [0.90, 0.90, 0.92],  # white at horizon
])

_PALETTES = np.stack([_PALETTE_DAWN, _PALETTE_EARLY, _PALETTE_MID, _PALETTE_NOON])
_PALETTE_Y_KEYS = np.array([0.0, 0.3, 0.7, 1.0])

# Row fractions where each colour stop sits (same for every palette)
_STOP_ROWS = np.array([0.0, 0.33, 0.66, 1.0])


def _interpolate_palette(sun_y, sun_x):
    """Return a (4, 3) palette interpolated for the given sun_y, then
    warm-shifted by sun_x."""
    # Piecewise-linear interp between the 4 reference palettes
    idx = np.searchsorted(_PALETTE_Y_KEYS, sun_y, side="right") - 1
    idx = np.clip(idx, 0, len(_PALETTE_Y_KEYS) - 2)
    t = (sun_y - _PALETTE_Y_KEYS[idx]) / (_PALETTE_Y_KEYS[idx + 1] - _PALETTE_Y_KEYS[idx])
    palette = (1 - t) * _PALETTES[idx] + t * _PALETTES[idx + 1]

    # sun_x warmth shift: push towards orange/gold, boost saturation
    warm_shift = np.array([0.08, 0.02, -0.06])  # add red/green, remove blue
    palette = palette + warm_shift * (sun_x - 0.5) * 2  # centred at 0.5
    return np.clip(palette, 0, 1)


def render_image(sun_x, sun_y, size=32):
    """Render a single sunrise image as (size, size, 3) float32 in [0,1]."""
    img = np.zeros((size, size, 3), dtype=np.float32)

    palette = _interpolate_palette(sun_y, sun_x)

    # ---- 1. Sky gradient ----
    row_fracs = np.linspace(0, 1, size).reshape(-1, 1)  # (H, 1)
    # Interpolate across the 4 colour stops
    for i in (0, 1, 2):
        mask = (row_fracs >= _STOP_ROWS[i]) & (row_fracs <= _STOP_ROWS[i + 1])
        t = (row_fracs - _STOP_ROWS[i]) / (_STOP_ROWS[i + 1] - _STOP_ROWS[i])
        blend = (1 - t) * palette[i] + t * palette[i + 1]  # (H, 3)
        # broadcast across width
        for c in (0, 1, 2):
            ch = blend[:, c:c+1] * np.ones((1, size))
            img[:, :, c] = np.where(mask, ch, img[:, :, c])

    # ---- 2. Sun disc (Gaussian blob) ----
    sx = sun_x * size
    sy = (1 - sun_y) * size * 0.7 + 0.1 * size
    radius = size * 0.12  # ~3.8 px at 32

    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    dist2 = (xx - sx) ** 2 + (yy - sy) ** 2
    sun_blob = np.exp(-dist2 / (2 * radius ** 2))

    # Sun colour depends on elevation
    if sun_y < 0.3:
        sun_color = np.array([1.0, 0.45, 0.15])  # deep orange
    elif sun_y < 0.7:
        t = (sun_y - 0.3) / 0.4
        sun_color = (1 - t) * np.array([1.0, 0.65, 0.2]) + t * np.array([1.0, 0.9, 0.5])
    else:
        t = (sun_y - 0.7) / 0.3
        sun_color = (1 - t) * np.array([1.0, 0.9, 0.5]) + t * np.array([1.0, 0.98, 0.85])

    for c in (0, 1, 2):
        img[:, :, c] = img[:, :, c] + sun_blob * sun_color[c] * 1.2

    # ---- 3. Atmospheric glow ----
    glow_radius = size * 0.5
    glow = np.exp(-dist2 / (2 * glow_radius ** 2))
    glow_intensity = 0.4 * max(0, 1 - sun_y * 1.5)  # stronger at low sun
    glow_color = np.array([0.9, 0.55, 0.15])
    for c in (0, 1, 2):
        img[:, :, c] = img[:, :, c] + glow * glow_color[c] * glow_intensity

    # ---- 4. Horizon silhouette ----
    horizon_base = int(size * 0.85)
    # gentle hills via sine
    phase = sun_x * 2 * math.pi
    hill_y = np.sin(np.linspace(0, 2 * math.pi, size) + phase) * size * 0.03
    for col, hill in enumerate(hill_y):
        h = int(horizon_base + hill)
        h = max(0, min(size - 1, h))
        # dark land colour: blend dark blue-black to dark green-brown by sun_x
        land_a = np.array([0.04, 0.04, 0.10])
        land_b = np.array([0.10, 0.16, 0.10])
        land_color = (1 - sun_x) * land_a + sun_x * land_b
        img[h:, col, :] = land_color

    # ---- 5. Noise & clip ----
    img += np.random.normal(0, 0.01, img.shape).astype(np.float32)
    np.clip(img, 0, 1, out=img)

    return img

## 2. Three thousand sunrises, and their seasons

We pick $3000$ moments at random, a `sun_x` and a `sun_y` for each, and draw a sunrise for each moment. The network learns from the first $2400$, the **training** sunrises. The last $600$ are kept apart as **test** sunrises: the network never sees them while it learns, so they show how well it does on pictures it has not seen.

The season comes from the time of the year, cut into three equal parts: **winter** for `sun_x` below $1/3$, **spring or autumn** between $1/3$ and $2/3$, **summer** above $2/3$. Spring and autumn are one class because a sunrise cannot tell them apart. The sun rises at the same point of the horizon in late March and in late September, and the generator, like the sky, has one number for the time of the year, from winter to summer. A spring sunrise and an autumn sunrise are the same picture.

In [ ]:
#@title Three thousand sunrises { display-mode: "form" }
np.random.seed(0)   # seeded: the same moments, and the same pictures, on every computer

# 3000 moments, one row (sun_x, sun_y) each, both between 0 and 1, and a sunrise for each.
sun = np.random.uniform(0, 1, size=(3000, 2))
pictures = np.array([render_image(sun_x, sun_y) for sun_x, sun_y in sun])   # 3000 x 32 x 32 x 3
pictures = torch.tensor(pictures).permute(0, 3, 1, 2).contiguous()        # PyTorch wants the colours first

# The season from the time of the year, cut into three equal parts.
SEASONS = ["winter", "spring or autumn", "summer"]
seasons = torch.tensor(np.floor(3 * sun[:, 0]).astype(int))   # 0, 1 or 2

# The first 2400 sunrises to train on, the last 600 to test on.
train_pictures, test_pictures = pictures[:2400], pictures[2400:]
train_seasons, test_seasons = seasons[:2400], seasons[2400:]
print(len(train_pictures), "training sunrises and", len(test_pictures), "test sunrises,",
      "each", tuple(train_pictures.shape[1:]), "numbers")

In [ ]:
#@title Four sunrises of each season { display-mode: "form" }
def show(ax, picture, title):
    ax.imshow(picture.permute(1, 2, 0))   # the colours last again, for drawing
    ax.set(xticks=[], yticks=[], title=title)



# The first four training sunrises of each season, one season per row.
picks = [i for season in (0, 1, 2) for i in np.flatnonzero(train_seasons.numpy() == season)[:4]]
fig, axes = plt.subplots(3, 4, figsize=(10, 8.4))
for ax, i in zip(axes.flat, picks):
    show(ax, train_pictures[i], SEASONS[train_seasons[i]])
plt.show()

## 3. The network

The network has two parts.

* Its **body** reads the picture with two **convolution blocks**, one line each. `nn.Conv2d` slides small filters of $3 \times 3$ pixels over the picture, as on weekend 3, and each filter makes a new picture, a **map**; `padding=1` keeps the map as wide and as high as the picture. `nn.ReLU()` turns every negative number into $0$. `nn.MaxPool2d(2)` keeps the largest number of every square of $2 \times 2$, which halves the width and the height. The first block turns the $3$ colours of $32 \times 32$ pixels into $8$ maps of $16 \times 16$, the second turns those into $16$ maps of $8 \times 8$. Then `nn.Flatten()` lays them out as one list of $16 \times 8 \times 8 = 1024$ numbers.
* Its **head** turns the $1024$ numbers into the answer: $3$ numbers, one **score** per season. The season with the highest score is the network's answer.

In [ ]:
torch.manual_seed(0)   # seeded: the same starting weights on every computer

# The body reads the picture: two convolution blocks, one line each.
body = nn.Sequential(
    nn.Conv2d(3, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 3 x 32 x 32 into 8 x 16 x 16
    nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 8 x 16 x 16 into 16 x 8 x 8
    nn.Flatten(),                                                             # 16 x 8 x 8 = 1024 numbers in one list
)

# The head gives the answer: 3 scores, one per season.
head = nn.Linear(16 * 8 * 8, 3)

model = nn.Sequential(body, head)

## 4. Training

The loop of weekend 1: while the loss is too big, take a step. The three scores of a picture become three probabilities that add up to $1$, and the loss of one picture is the **cross-entropy** $\mathscr{L}_i = -\log p_i$, where $p_i$ is the probability that the network gives to the picture's true season. It is $0$ when the network is sure and right, and it grows without limit when the network is sure and wrong. `nn.functional.cross_entropy` turns the scores into probabilities and takes the logarithm in one call.

The loss $\bar{\mathscr{L}}$ is the average over $64$ training sunrises picked at random. The loop stops once it is below $0.01$: then a typical picture of the batch gets a probability of about $0.99$ for its true season, since $-\log 0.99$ is about $0.01$.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)   # lr: how big a step is

loss = 1.0                                                     # any start above the threshold
while loss > 0.01:
    batch = torch.randint(0, len(train_pictures), (64,))       # 64 training sunrises picked at random
    scores = model(train_pictures[batch])                      # 3 scores for each of them
    loss = nn.functional.cross_entropy(scores, train_seasons[batch])   # the average of -log p
    optimizer.zero_grad()                                      # forget the gradients of the previous step
    loss.backward()                                            # compute the gradient of the loss
    optimizer.step()                                           # move every weight a small step against its gradient
    print(f"\rcross-entropy: {loss.item():.4f}", end="")

## 5. How well does it do?

The folded cell below asks the network for the season of the $600$ test sunrises and counts how often it is right. The table says how often each true season, in a row, got each answer, in a column. The plot puts every test sunrise at its time of the year, across, and its time of the day, up: a dot is a right answer, a cross a wrong one. Look at where the crosses sit. Last come a few of the test sunrises it got wrong, with their `sun_x`.

In [ ]:
#@title The season of the 600 test sunrises { display-mode: "form" }
with torch.no_grad():                                  # no training here, so no gradients
    answers = model(test_pictures).argmax(dim=1)       # the season with the highest score
right = (answers == test_seasons).numpy()
print(f"right on {right.sum()} of {len(right)} test sunrises: {right.mean():.1%}")

# How often each true season (a row) got each answer (a column).
print()
print(" " * 18 + "".join(f"{name:>18}" for name in SEASONS))
for true, name in zip((0, 1, 2), SEASONS):
    counts = [int(((test_seasons == true) & (answers == said)).sum()) for said in (0, 1, 2)]
    print(f"{name:>18}" + "".join(f"{count:>18}" for count in counts))

test_sun = sun[2400:]
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(test_sun[right, 0], test_sun[right, 1], s=12, color="#0072B2", marker="o", label="right")
ax.scatter(test_sun[~right, 0], test_sun[~right, 1], s=70, color="#D55E00", marker="x", label="wrong")
ax.axvline(1 / 3, color="grey", linestyle="--")
ax.axvline(2 / 3, color="grey", linestyle="--")
for middle, name in zip((1 / 6, 1 / 2, 5 / 6), SEASONS):
    ax.text(middle, 1.04, name, ha="center", fontsize=12)
ax.set(xlim=(0, 1), ylim=(0, 1), xticks=[0, 1 / 3, 2 / 3, 1], xticklabels=["0", "1/3", "2/3", "1"],
       xlabel="time of the year, sun_x: winter at 0, summer at 1",
       ylabel="time of the day, sun_y: dawn at 0, noon at 1")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.show()

wrong = np.flatnonzero(~right)
if len(wrong) > 0:
    fig, axes = plt.subplots(1, 6, figsize=(17, 4))
    for ax in axes:
        ax.axis("off")
    for ax, i in zip(axes, wrong):
        show(ax, test_pictures[i], f"true: {SEASONS[test_seasons[i]]}\nsaid: {SEASONS[answers[i]]}\nsun_x = {test_sun[i, 0]:.3f}")
    plt.show()

## The lines the exercise asks for

* **The body**: the two lines with `nn.Conv2d` in part 3. The exercise uses the same body.
* **The head**: here `nn.Linear(16 * 8 * 8, 3)`, three scores, one per season. A time of the day is one number.
* **The loss**: here `nn.functional.cross_entropy`, which is made for choosing one of several classes. The miss of a number is measured as on weekend 1: the squared difference between the predicted and the true number, averaged, `((prediction - truth) ** 2).mean()`.
* **One training step**: `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`, the same three calls in every training loop.